# Silver Layer - Asset Reference Transformation

**Purpose**: Transform bronze reference tables into stable silver_asset_reference lookup structure with:
- Standardized asset identifiers (substation_id)
- Clean region values and country codes
- Voltage and capacity classifications
- Asset age calculations
- Enriched with region details
- Reliable join keys for fact tables

**Input**: 
- `vattenfall_dev.raw.ref_substations` (50 substations)
- `vattenfall_dev.raw.ref_regions` (11 regions)

**Output**: `vattenfall_dev.refined.silver_asset_reference`

**Key Transformations**:
1. Validate and standardize asset identifiers
2. Extract country codes from region codes
3. Classify voltage levels and capacity
4. Calculate asset age and status
5. Enrich with region details (left join)
6. Create reliable join keys (asset_key, region_key, country_key)
7. Add data quality indicators

In [0]:
# Setup Python path to access project modules
import sys
project_root = "/Workspace/Users/gharbi@startsteps.org/vattenfall-capstone-project"
if project_root not in sys.path:
    sys.path.append(project_root)

# Import transformation modules
from src.transforms import asset_reference_transforms
import importlib

# Reload module to pick up latest changes
importlib.reload(asset_reference_transforms)

print("✓ Setup complete")
print(f"✓ Project root: {project_root}")
print(f"✓ Module loaded: asset_reference_transforms")

In [0]:
# Load substations and regions reference tables
df_substations = spark.table("vattenfall_dev.raw.ref_substations")
df_regions = spark.table("vattenfall_dev.raw.ref_regions")

print("Bronze Reference Data Loaded:")
print(f"\nSubstations:")
print(f"  Records: {df_substations.count():,}")
print(f"  Columns: {len(df_substations.columns)}")

print(f"\nRegions:")
print(f"  Records: {df_regions.count():,}")
print(f"  Columns: {len(df_regions.columns)}")

print("\nSubstations Schema:")
df_substations.printSchema()

print("\nSample Substations:")
display(df_substations.limit(5))

print("\nSample Regions:")
display(df_regions.limit(5))

In [0]:
from pyspark.sql import functions as F

print("=== Reference Data Quality Analysis ===")

# Check for duplicates in substations
print("\n1. Duplicate Check:")
dup_count = df_substations.groupBy("substation_id").count().filter(F.col("count") > 1).count()
print(f"   Duplicate substation_ids: {dup_count}")

# Region code distribution
print("\n2. Region Code Distribution:")
display(df_substations.groupBy("region_code").count().orderBy("region_code"))

# Voltage distribution
print("\n3. Voltage Distribution:")
display(df_substations.groupBy("voltage_kv").count().orderBy("voltage_kv"))

# Capacity statistics
print("\n4. Capacity Statistics:")
stats = df_substations.select(
    F.min("capacity_mva").alias("min_capacity"),
    F.max("capacity_mva").alias("max_capacity"),
    F.avg("capacity_mva").alias("avg_capacity"),
    F.count("*").alias("total_substations")
)
display(stats)

# Commissioned year distribution
print("\n5. Age Distribution (by decade):")
decade_dist = df_substations.withColumn(
    "decade",
    F.concat((F.floor(F.col("commissioned_year") / 10) * 10).cast("string"), F.lit("s"))
).groupBy("decade").count().orderBy("decade")
display(decade_dist)

In [0]:
# Apply the complete transformation pipeline
df_silver = asset_reference_transforms.transform_to_silver_asset_reference(
    df_substations, 
    df_regions
)

print("Silver Asset Reference Transformed:")
print(f"  Records: {df_silver.count():,}")
print(f"  Columns: {len(df_silver.columns)}")

print("\nNew Columns Added:")
bronze_cols = set(df_substations.columns)
silver_cols = set(df_silver.columns)
new_cols = sorted(silver_cols - bronze_cols)
for col in new_cols:
    print(f"  - {col}")

print(f"\nTotal new columns: {len(new_cols)}")

print("\nSilver Schema:")
df_silver.printSchema()

In [0]:
# Display sample of transformed silver data
print("=== Silver Asset Reference Sample ===")
display(df_silver.limit(10))

print("\n=== Key Transformations Verification ===")

print("\n1. Country Codes Extracted:")
display(df_silver.groupBy("country_code", "country_name").count().orderBy("country_code"))

print("\n2. Voltage Level Classification:")
display(df_silver.groupBy("voltage_level", "voltage_kv").count().orderBy("voltage_level", "voltage_kv"))

print("\n3. Capacity Categories:")
display(df_silver.groupBy("capacity_category").count().orderBy("capacity_category"))

print("\n4. Asset Age Categories:")
display(df_silver.groupBy("asset_age_category").count().orderBy(
    F.when(F.col("asset_age_category") == "new", 1)
     .when(F.col("asset_age_category") == "modern", 2)
     .when(F.col("asset_age_category") == "mature", 3)
     .when(F.col("asset_age_category") == "aging", 4)
     .otherwise(5)
))

print("\n5. Join Keys Verification:")
display(df_silver.select("asset_key", "region_key", "country_key", "substation_id", "region_code", "country_code").limit(5))

print("\n6. Data Quality Scores:")
display(df_silver.groupBy("data_quality_score").count().orderBy("data_quality_score"))

In [0]:
# Verify that region details were properly joined
print("=== Region Enrichment Verification ===")

print("\n1. Records with region details:")
with_region = df_silver.filter(F.col("region_name").isNotNull()).count()
total = df_silver.count()
print(f"   Records with region_name: {with_region}/{total} ({with_region/total*100:.1f}%)")

print("\n2. Sample enriched records:")
display(df_silver.select(
    "substation_id",
    "region_code",
    "region_name",
    "country_name",
    "region_population",
    "region_area_km2"
).limit(10))

print("\n3. Missing region details (if any):")
missing_region = df_silver.filter(F.col("region_name").isNull())
missing_count = missing_region.count()
if missing_count > 0:
    print(f"   ⚠️  {missing_count} substations missing region details")
    display(missing_region.select("substation_id", "region_code", "region_name").limit(10))
else:
    print(f"   ✓ All substations have region details")

In [0]:
# Write the silver dataframe to the refined schema
table_name = "vattenfall_dev.refined.silver_asset_reference"

df_silver.write \
    .mode("overwrite") \
    .format("delta") \
    .option("overwriteSchema", "true") \
    .saveAsTable(table_name)

print(f"✓ Silver table written successfully: {table_name}")

# Verify the written table
df_verify = spark.table(table_name)
print(f"\nVerification:")
print(f"  Records in table: {df_verify.count():,}")
print(f"  Columns in table: {len(df_verify.columns)}")

# Show table location and properties
print(f"\nTable Properties:")
spark.sql(f"DESCRIBE EXTENDED {table_name}").filter(
    F.col("col_name").isin(["Location", "Provider", "Type"])
).show(truncate=False)

In [0]:
# Final summary of the silver asset reference transformation
df_bronze_subs = spark.table("vattenfall_dev.raw.ref_substations")
df_silver = spark.table("vattenfall_dev.refined.silver_asset_reference")

bronze_count = df_bronze_subs.count()
silver_count = df_silver.count()
bronze_cols = len(df_bronze_subs.columns)
silver_cols = len(df_silver.columns)
filtered_count = bronze_count - silver_count

print("="*60)
print("SILVER ASSET REFERENCE TRANSFORMATION SUMMARY")
print("="*60)
print(f"\n📊 Record Counts:")
print(f"   Bronze Substations: {bronze_count:>6,}")
print(f"   Silver Assets:      {silver_count:>6,}")
print(f"   Filtered Out:       {filtered_count:>6,} ({filtered_count/bronze_count*100:.1f}%)")
print(f"\n📋 Column Counts:")
print(f"   Bronze Columns:     {bronze_cols:>6}")
print(f"   Silver Columns:     {silver_cols:>6}")
print(f"   Columns Added:      {silver_cols - bronze_cols:>6}")
print(f"\n✅ Key Transformations:")
print(f"   • Asset identifiers standardized (uppercase, trimmed)")
print(f"   • Country codes extracted from region codes")
print(f"   • Voltage levels classified (extra_high/high/medium/low)")
print(f"   • Capacity categories assigned (very_large/large/medium/small)")
print(f"   • Asset age calculated and categorized")
print(f"   • Region details enriched via left join")
print(f"   • Join keys created (asset_key, region_key, country_key)")
print(f"   • Data quality scores calculated")
print(f"   • Operational status indicators added")
print(f"\n🔑 Reliable Join Keys:")
print(f"   • asset_key: Primary key for joining with grid events")
print(f"   • region_key: For region-based analytics")
print(f"   • country_key: For country-level aggregations")
print(f"\n📍 Table Location: vattenfall_dev.refined.silver_asset_reference")
print("="*60)